In [1]:
import os
os.chdir('/home/smallyan/eval_agent')

# Load environment from bashrc  
bashrc_path = os.path.expanduser('~/.bashrc')
with open(bashrc_path, 'r') as f:
    for line in f:
        line = line.strip()
        if line.startswith('export '):
            parts = line[7:].split('=', 1)
            if len(parts) == 2:
                key = parts[0]
                value = parts[1].strip('"').strip("'")
                os.environ[key] = value

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"HF_HOME: {os.environ.get('HF_HOME', 'Not set')}")

CUDA available: True
HF_HOME: /net/projects2/chai-lab/shared_models


# Generalizability Evaluation for Linear Relational Embeddings (LRE)

## Repository: `/net/scratch2/smallyan/relations_eval`

This notebook evaluates whether the Linear Relational Embedding (LRE) findings generalize:

1. **GT1**: Generalization to a New Model
2. **GT2**: Generalization to New Data
3. **GT3**: Method / Specificity Generalizability

## Background

The LRE paper ("Linearity of Relation Decoding in Transformer LMs") demonstrates that:
- Relational knowledge can be decoded via linear transformations: LRE(s) = Ws + b
- This was tested on GPT-J, GPT-2-XL, and LLaMA-13B
- The method achieves >60% faithfulness on 48% of relations

In [2]:
import sys
repo_path = '/net/scratch2/smallyan/relations_eval'
sys.path.insert(0, repo_path)
os.chdir(repo_path)

from src import data, functional
import json
print("Source modules loaded")

# Load a sample relation
relation_file = f'{repo_path}/data/factual/country_capital_city.json'
with open(relation_file, 'r') as f:
    relation_data = json.load(f)
print(f"Loaded relation: {relation_data['name']}")
print(f"Prompt template: {relation_data['prompt_templates'][0]}")
print(f"Number of samples: {len(relation_data['samples'])}")

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Source modules loaded
Loaded relation: country capital city
Prompt template: The capital city of {} is
Number of samples: 24


---

# GT1: Model Generalization Test

**Goal**: Test if the LRE finding generalizes to a NEW model not used in original work.

**Original models used**: GPT-J-6B, GPT-2-XL, LLaMA-13B

**New model to test**: Pythia-2.8B (EleutherAI/pythia-2.8b)
- Similar architecture to GPT-J (GPT-NeoX based)
- NOT used in the original paper
- Available in cache

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch.nn.functional as F

device = torch.device("cuda")

# Load Pythia-2.8B
model_name = "EleutherAI/pythia-2.8b"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype=torch.float16,
    device_map="cuda"
)
model.eval()

print(f"Model loaded successfully")
print(f"Num layers: {model.config.num_hidden_layers}")
print(f"Hidden size: {model.config.hidden_size}")

Loading EleutherAI/pythia-2.8b...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

In [4]:
# Test basic inference
print(f"Model loaded: {type(model).__name__}")
print(f"Layers: {model.config.num_hidden_layers}")

test_prompt = "The capital city of France is"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=3, do_sample=False)
    
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: '{test_prompt}'")
print(f"Output: '{result}'")

In [5]:
# Check if model is accessible
print("Model check:")
print(f"Model type: {type(model)}")
print(f"Device: {next(model.parameters()).device}")

In [6]:
print("testing")